# Laya vs Logistic Regression — ticket type
Can a pretrained decision model route tickets better than the bag-of-words baseline? The baseline (TF-IDF → Logistic Regression, [ticket classifier](../topic-classification/department_classifier_pipeline.ipynb)) reached **79.9%** test accuracy and is at its ceiling on this data. [Laya](https://huggingface.co/convaiinnovations/laya) (ModernBERT-large, 421M parameters) reads the whole ticket and answers a typed question with a probability for each answer.

**Arms**, all on the same rows, same target (`type`), same input (`subject` + `body`):
- **A** — Logistic Regression baseline, frozen (not retrained)
- **B** — Laya zero-shot on **raw** text
- **C** — Laya zero-shot on the baseline's **cleaned** text
- **D** — Laya fine-tuned; raw or cleaned text is decided by B vs C (next notebook step)

**Rules (fixed before any Laya score):** validation only — the test set stays sealed until D is chosen · cleaning matters only if B and C differ by more than 1 point of accuracy · the question wording is written once and never tuned on validation.

---
# Phase 1 — Same rows as the baseline

## Step 1 — Setup: seed, constants, library versions

In [1]:
# Step 1 — imports, seed, constants
import sys
import time
from pathlib import Path

import joblib
import laya
import numpy as np
import pandas as pd
import sklearn
import torch
import transformers
from sklearn.metrics import accuracy_score, f1_score, recall_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

BASELINE_DIR = Path("../topic-classification")
RAW_FILE = BASELINE_DIR / "data" / "aa_dataset-tickets-multi-lang-5-2-50-version.csv"
SPLITS_FILE = BASELINE_DIR / "artifacts" / "splits.csv"            # written by the baseline's Phase 4B (export) step
BASELINE_MODEL = BASELINE_DIR / "artifacts" / "type_classifier.joblib"
PRED_DIR = Path("predictions")
PRED_DIR.mkdir(exist_ok=True)

MODEL_ID = "convaiinnovations/laya"
TYPES = np.array(["Incident", "Request", "Problem", "Change"])   # same order as the baseline
TIE_TOLERANCE = 0.01          # differences within 1 point of accuracy are noise (baseline rule)
ROUTE_THRESHOLD = 0.75        # the baseline's auto-routing line
TARGET_AUTO_ACCURACY = 0.95   # product target for tickets routed without a person
BASELINE_VAL = {"accuracy": 0.789632, "macro F1": 0.807491}   # baseline notebook, Phase 4B (confirm)

for lib in [np, pd, sklearn, torch, transformers, laya]:
    print(f"{lib.__name__:13s} {lib.__version__}")
print(f"{'python':13s} {sys.version.split()[0]}   ·   MPS available: {torch.backends.mps.is_available()}")

/Users/ikarthik/Documents/GitHub/ReLearning-Python/laya-discover/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


numpy         2.5.3
pandas        3.0.6
sklearn       1.9.1
torch         2.14.1
transformers  5.18.0
laya          0.3.28
python        3.12.15   ·   MPS available: True


## Step 2 — Load the baseline's split, add the raw text
**Decision:** the split is read, never recomputed. `text_raw` is `subject + body` joined exactly as the baseline does it *before* cleaning; `text_clean` is the baseline's cleaned text (formatting leftovers removed, emails / URLs / long numbers replaced).

In [2]:
# Step 2 — tickets = baseline split + raw text; Seam 1 asserts stop here if the rows differ
df_raw = pd.read_csv(RAW_FILE)
splits = pd.read_csv(SPLITS_FILE)

tickets = splits.join(df_raw[["subject", "body", "type", "language"]], on="raw_row", rsuffix="_raw")
tickets["text_raw"] = (tickets["subject"].fillna("") + " " + tickets["body"].fillna("")).str.strip()

assert (tickets["type"] == tickets["type_raw"]).all(), "split file does not line up with the raw CSV"
assert (tickets["language"] == "en").all(), "split file holds non-English rows"
assert set(tickets["type"]) == set(TYPES)
for a, b in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    fa, fb = (set(tickets.loc[tickets["split"] == s, "family_id"]) for s in (a, b))
    assert not fa & fb, f"a family appears in both {a} and {b}"
assert tickets["split"].value_counts().to_dict() == {"train": 11670, "validation": 2334, "test": 2334}
tickets = tickets[["raw_row", "family_id", "split", "type", "text_raw", "text_clean"]]
print("Same rows, same families, same split as the baseline ✓")

changed = tickets["text_raw"] != tickets["text_clean"]
print(f"Cleaning changed the text of {changed.sum():,} of {len(tickets):,} tickets ({changed.mean():.0%})\n")
example = tickets[changed & tickets["text_raw"].str.contains(r"\\n|<br|@|http", regex=True)].iloc[0]
print(f"raw:   {example['text_raw'][:300]}\nclean: {example['text_clean'][:300]}")
val = tickets[tickets["split"] == "validation"].reset_index(drop=True)

Same rows, same families, same split as the baseline ✓
Cleaning changed the text of 1,086 of 16,338 tickets (7%)

raw:   Account Disruption Dear Customer Support Team,\n\nI am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in m
clean: Account Disruption Dear Customer Support Team, I am writing to report a significant problem with the centralized account management portal, which currently appears to be offline. This outage is blocking access to account settings, leading to substantial inconvenience. I have attempted to log in mult


---
# Phase 2 — One scoring step for every arm
**How arms are judged:** accuracy decides · arms within 1 point are tied → higher macro F1 · every type must have recall > 0. Calibration (ECE) and auto-routing are reported next to them, because a router acts on confidence, not only on the top answer.

Every arm writes the same prediction file — `raw_row, split, type, pred, conf, p_Incident … p_Change` — and one function scores them all.

In [3]:
# Phase 2 — prediction file contract (Seam 2) and the one scoring function
P_COLS = [f"p_{t}" for t in TYPES]

def check_predictions(pred, rows):
    assert list(pred.columns[:5]) == ["raw_row", "split", "type", "pred", "conf"] and set(P_COLS) <= set(pred.columns)
    assert (pred["raw_row"].to_numpy() == rows["raw_row"].to_numpy()).all(), "prediction rows differ from the split rows"
    assert set(pred["pred"]) <= set(TYPES)
    assert np.allclose(pred[P_COLS].sum(axis=1), 1, atol=1e-3), "probabilities do not add up to 1"
    assert np.allclose(pred["conf"], pred[P_COLS].max(axis=1)) and (pred["pred"] == TYPES[pred[P_COLS].to_numpy().argmax(1)]).all()

def expected_calibration_error(conf, correct, bins=10):  # same as the baseline, Step 19
    bin_id = np.minimum((conf * bins).astype(int), bins - 1)
    return sum(abs(correct[bin_id == b].mean() - conf[bin_id == b].mean()) * (bin_id == b).mean()
               for b in range(bins) if (bin_id == b).any())

def score(pred):
    y, p, conf = pred["type"].to_numpy(), pred["pred"].to_numpy(), pred["conf"].to_numpy()
    correct = y == p
    recalls = recall_score(y, p, labels=TYPES, average=None, zero_division=0)
    thresholds = np.round(np.arange(0.40, 0.96, 0.05), 2)
    meets = [t for t in thresholds if (conf >= t).any() and correct[conf >= t].mean() >= TARGET_AUTO_ACCURACY]
    own = min(meets) if meets else np.nan
    return pd.Series({
        "accuracy": accuracy_score(y, p), "macro F1": f1_score(y, p, labels=TYPES, average="macro"),
        **{f"recall {t}": r for t, r in zip(TYPES, recalls)},
        "ECE": expected_calibration_error(conf, correct),
        f"auto-routed @{ROUTE_THRESHOLD}": (conf >= ROUTE_THRESHOLD).mean(),
        f"accuracy @{ROUTE_THRESHOLD}": correct[conf >= ROUTE_THRESHOLD].mean() if (conf >= ROUTE_THRESHOLD).any() else np.nan,
        f"own line for {TARGET_AUTO_ACCURACY:.0%}": own,
        "auto-routed @own line": (conf >= own).mean() if meets else 0.0,
    })

def save_predictions(arm, rows, proba):
    proba = np.asarray(proba)
    pred = pd.DataFrame({"raw_row": rows["raw_row"], "split": rows["split"], "type": rows["type"],
                         "pred": TYPES[proba.argmax(1)], "conf": proba.max(1)})
    pred[P_COLS] = proba
    check_predictions(pred, rows)
    pred.to_csv(PRED_DIR / f"{arm}_{rows['split'].iloc[0]}.csv", index=False)
    return pred

## Arm A — the baseline, rebuilt from its saved model (scoring check)
The saved pipeline was trained on the training rows only, so its validation score must equal the baseline notebook's. If it does, the scoring step is proven correct.

In [4]:
# Arm A — baseline on validation; must reproduce the baseline notebook's numbers
lr = joblib.load(BASELINE_MODEL)
t0 = time.perf_counter()
proba_a = pd.DataFrame(lr.predict_proba(val["text_clean"]), columns=lr.classes_)[TYPES]
sec_per_ticket = {"A · Logistic Regression": (time.perf_counter() - t0) / len(val)}
pred_a = save_predictions("arm_a_lr", val, proba_a)
score_a = score(pred_a)

for k, v in BASELINE_VAL.items():
    assert abs(score_a[k] - v) < 5e-4, f"{k}: {score_a[k]:.6f} here vs {v} in the baseline notebook"
print(f"Scoring check ✓ — accuracy {score_a['accuracy']:.4f}, macro F1 {score_a['macro F1']:.4f} (baseline notebook: "
      f"{BASELINE_VAL['accuracy']:.4f}, {BASELINE_VAL['macro F1']:.4f})")

Scoring check ✓ — accuracy 0.7896, macro F1 0.8075 (baseline notebook: 0.7896, 0.8075)


---
# Phase 3 — Laya zero-shot
**The question (written once, from the usual IT-service meaning of each type — never tuned on validation):**

In [5]:
# Phase 3 — the one typed question Laya answers, and the batch predictor
TYPE_QUESTION = {"ticket_type": {"type": "choice",
    "instructions": "What type of IT support ticket is this?",
    "criteria": {
        "Incident": "something is broken, down or not working right now and needs a fix",
        "Request":  "asks for information, access, help or a standard service; nothing is broken",
        "Problem":  "a recurring or underlying cause behind repeated failures that needs investigation",
        "Change":   "asks to add, upgrade, modify or improve a system, feature or configuration"}}}

agent = laya.load(MODEL_ID)
n_params = sum(p.numel() for p in agent.model.parameters())
print(f"{MODEL_ID}: {n_params / 1e6:,.0f}M parameters on {agent.device}")

def predict_laya(texts, batch_size=32):
    # batches of 5+ rows run in fp16 on MPS: probabilities differ from single calls by < 0.001, the answer does not change
    out = agent.predict_batch(list(texts), TYPE_QUESTION, batch_size=batch_size, sort_by_length=True)
    proba = [[r["answers"]["ticket_type"]["probabilities"][t] for t in TYPES] for r in out]
    truncated = np.array([r["usage"]["truncated"] for r in out])
    return np.array(proba), truncated

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 2187.27it/s]

/Users/ikarthik/Documents/GitHub/ReLearning-Python/laya-discover/.venv/lib/python3.12/site-packages/laya/agent.py:2125: RuntimeWarning: laya: this checkpoint ships invalid temperatures or values outside [0.5, 5]; using choice:11+=0.10058280825614929 -> 0.5. Treat confidence from the affected entries as uncalibrated.
  return Agent(model_id_or_path, device=device, token=token, subfolder=subfolder, fast=fast,


convaiinnovations/laya: 421M parameters on mps


## Smoke test — 50 validation tickets, before the full run

In [6]:
# Smoke test — the run completes and the prediction file passes the contract
smoke = val.head(50)
proba, _ = predict_laya(smoke["text_raw"])
save_predictions("smoke", smoke, proba)
(PRED_DIR / "smoke_validation.csv").unlink()
print("Smoke test ✓ — 50 tickets predicted, prediction file valid")

Smoke test ✓ — 50 tickets predicted, prediction file valid


## Arms B and C — Laya zero-shot on raw and on cleaned text (validation)

In [7]:
# Arms B and C — same model, same question; only the text differs
preds = {"A · Logistic Regression": pred_a}
for arm, name, col in [("arm_b_zeroshot_raw", "B · Laya zero-shot, raw", "text_raw"),
                       ("arm_c_zeroshot_clean", "C · Laya zero-shot, cleaned", "text_clean")]:
    t0 = time.perf_counter()
    proba, truncated = predict_laya(val[col])
    sec_per_ticket[name] = (time.perf_counter() - t0) / len(val)
    preds[name] = save_predictions(arm, val, proba)
    print(f"{name}: {len(val):,} tickets in {sec_per_ticket[name] * len(val) / 60:.1f} min · "
          f"cut at 512 tokens: {truncated.sum():,}")

B · Laya zero-shot, raw: 2,334 tickets in 3.3 min · cut at 512 tokens: 0


C · Laya zero-shot, cleaned: 2,334 tickets in 3.9 min · cut at 512 tokens: 0


---
# Phase 4 — Compare on validation

In [8]:
# Phase 4 — one table, every arm scored by the same function
comparison = pd.DataFrame({name: score(p) for name, p in preds.items()}).T
comparison["seconds / ticket"] = pd.Series(sec_per_ticket)
comparison["model size"] = ["%.1f MB" % (BASELINE_MODEL.stat().st_size / 1e6)] + [f"{n_params / 1e6:,.0f}M params"] * 2
comparison.round(3)

,accuracy,macro F1,recall Incident,recall Request,recall Problem,recall Change,ECE,auto-routed @0.75,accuracy @0.75,own line for 95%,auto-routed @own line,seconds / ticket,model size
A · Logistic Regression,0.790,0.807,0.712,0.991,0.577,0.959,0.045,0.485,0.956,0.75,0.485,0.000,2.6 MB
"B · Laya zero-shot, raw",0.792,0.784,0.878,0.943,0.332,0.963,0.133,0.293,0.994,0.65,0.424,0.085,421M params
"C · Laya zero-shot, cleaned",0.789,0.783,0.870,0.943,0.334,0.963,0.131,0.292,0.994,0.65,0.421,0.100,421M params


In [9]:
# Phase 4 (cleaning verdict) — does the cleaned text change Laya's answers?
b, c = preds["B · Laya zero-shot, raw"], preds["C · Laya zero-shot, cleaned"]
gap = comparison.loc["C · Laya zero-shot, cleaned", "accuracy"] - comparison.loc["B · Laya zero-shot, raw", "accuracy"]
print(f"Same answer on raw and cleaned text: {(b['pred'] == c['pred']).mean():.1%} of tickets")
print(f"Accuracy, cleaned minus raw: {gap * 100:+.2f} points (noise band ±{TIE_TOLERANCE * 100:.0f})")
print("Verdict: cleaning " + ("MATTERS — Arm D trains on cleaned text" if gap > TIE_TOLERANCE
      else "does NOT matter — Arm D trains on raw text (nothing to build around the model)" if gap >= -TIE_TOLERANCE
      else "HURTS — Arm D trains on raw text"))

Same answer on raw and cleaned text: 99.7% of tickets
Accuracy, cleaned minus raw: -0.26 points (noise band ±1)
Verdict: cleaning does NOT matter — Arm D trains on raw text (nothing to build around the model)


In [10]:
# Phase 4 (where) — confusion of the better zero-shot arm, rows = true type, % of each row
from sklearn.metrics import confusion_matrix
best_zs = comparison.iloc[1:3]["accuracy"].idxmax()
pd.DataFrame(confusion_matrix(preds[best_zs]["type"], preds[best_zs]["pred"], labels=TYPES, normalize="true") * 100,
             index=TYPES, columns=TYPES).round(1).rename_axis(best_zs)

,Incident,Request,Problem,Change
"B · Laya zero-shot, raw",,,,
Incident,87.8,0.2,12.0,0.0
Request,4.8,94.3,0.2,0.8
Problem,66.6,0.2,33.2,0.0
Change,2.9,0.8,0.0,96.3


*Zero-shot Laya ties the baseline without one training example: **79.2%** vs **79.0%** accuracy on validation (inside the 1-point noise band). It gets there differently — Incident recall jumps (88% vs 71%) but Problem recall halves (33% vs 58%): two thirds of Problems are called Incident, the same blurred line the baseline found in the labels. Macro F1 is lower (0.78 vs 0.81) for the same reason. Laya is right 99% of the time when it is ≥ 75% sure, but it is that sure on fewer tickets (29% vs 49%), and its confidence is less honest (ECE 0.13 vs 0.05; the checkpoint warns that some of its temperatures are invalid). No ticket exceeds Laya's 512-token limit.*

### Decision: Arm D fine-tunes on raw text
Cleaning changes the answer on 0.3% of tickets and moves accuracy by −0.26 points — inside the noise. The model reads raw `subject + body`, so nothing needs to be built around it. **What fine-tuning has to show:** teach the dataset's own Incident ↔ Problem line, and make the confidence honest — by more than 1 point over the baseline, at a cost of ~0.16 s per ticket on a laptop GPU instead of ~0 s.

---
# Phase 5 — Arm D: Laya fine-tuned on the training rows
Training runs outside this notebook, in `train_arm_d.py` (Laya's official Apple-Silicon script, vendored; every change is listed in its header). This notebook only reads what the run wrote: one prediction file per epoch, scored on validation through the same `score()` as Arms A–C.

**How the training settings were reached (logs in `logs/`):**
- Script defaults diverged within ~12 updates: cross-entropy 0.96 → 2.51, worse than guessing among 4 types (ln 4 = 1.39) → **linear warmup** over the first 10% of updates.
- With warmup it still diverged at ~1,400 tickets. One-change-at-a-time runs, pass rule written first, each stopped at 2,400 tickets: no gradient checkpointing failed, learning rate × 0.3 failed, **cross-entropy only (Laya's RL term dropped)** passed.

**Run:** seed 42 · AdamW, learning rate 2.5e-5 encoder / 1e-4 head · warmup 146 updates then cosine to 1e-6 · 2 tickets × 16 accumulated = 32 per update · 4 epochs = 1,460 updates · fp32 on the Mac GPU (MPS) · raw text · Arm B's exact question.

In [11]:
# Phase 5 (load) — per-epoch validation predictions from the seed-42 run, plus the run log
import json, re
import train_arm_d
assert train_arm_d.TYPE_QUESTION == TYPE_QUESTION, "Arm D must be trained on Arm B's exact question"

SEED = 42
log_lines = [l for l in Path(f"logs/arm_d_seed{SEED}.log").read_text().splitlines() if "EPOCH" in l]
epoch_log = pd.DataFrame([{"epoch": int(m[1]), "train hours": float(m[2]), "train loss": float(m[3])}
                          for m in (re.search(r"EPOCH (\d+) done · ([\d.]+) h · \d+ updates · avg loss ([\d.]+)", l) for l in log_lines)]).set_index("epoch")

epoch_preds = {}
for e in epoch_log.index:
    p = pd.read_csv(PRED_DIR / f"arm_d_seed{SEED}_epoch{e}_validation.csv")
    check_predictions(p, val)
    epoch_preds[e] = p
curve = pd.concat([epoch_log, pd.DataFrame({e: score(p) for e, p in epoch_preds.items()}).T], axis=1)
curve[["train hours", "train loss", "accuracy", "macro F1", "recall Incident", "recall Problem", "ECE",
       f"auto-routed @{ROUTE_THRESHOLD}", f"accuracy @{ROUTE_THRESHOLD}"]].round(3)

,train hours,train loss,accuracy,macro F1,recall Incident,recall Problem,ECE,auto-routed @0.75,accuracy @0.75
1,2.16,0.365,0.764,0.798,0.592,0.674,0.098,0.723,0.858
2,2.08,0.216,0.784,0.809,0.686,0.584,0.123,0.856,0.825
3,2.13,0.056,0.799,0.810,0.774,0.485,0.178,0.965,0.809
4,2.41,0.003,0.796,0.809,0.770,0.480,0.189,0.979,0.804


*Training loss falls to ~0.003 — the model nearly memorises the training tickets — while validation accuracy levels off after epoch 3. Confidence gets less honest every epoch (ECE 0.10 → 0.19): by epoch 4 the model is ≥ 75% sure on 98% of tickets and right on 80% of them.*

In [12]:
# Phase 5 (choose) — the rule fixed before training: accuracy decides; within 1 point = tie → higher macro F1
best_acc = curve["accuracy"].max()
tied = curve[curve["accuracy"] >= best_acc - TIE_TOLERANCE]
CHOSEN_EPOCH = int(tied["macro F1"].idxmax())
print(f"Tied within 1 point of the best accuracy: epochs {list(tied.index)} → chosen: epoch {CHOSEN_EPOCH} "
      f"(macro F1 {tied.loc[CHOSEN_EPOCH, 'macro F1']:.4f})")

Tied within 1 point of the best accuracy: epochs [3, 4] → chosen: epoch 3 (macro F1 0.8097)


## Calibrate the chosen epoch on validation
**Decision rule (the baseline's):** keep a temperature only if it lowers ECE without raising log-loss, measured out of fold — 5 family-grouped folds on validation, so each fold's temperature is scored on tickets it was not fitted to.

**Fitted on raw logits, not on the saved probabilities.** Laya rounds probabilities to 4 decimals: on epoch 3, half of all validation tickets read exactly 1.0000, and a temperature fitted on those came out at the 5.0 cap. The logits were collected with Laya's own `laya.calibrate.records_from_labeled` and reproduce every saved answer.

In [13]:
# Phase 5 (calibrate) — temperature on raw logits; out-of-fold check; final T fitted on all validation
from sklearn.metrics import log_loss
from sklearn.model_selection import GroupKFold

logits = np.load(PRED_DIR / f"arm_d_seed{SEED}_epoch{CHOSEN_EPOCH}_validation_logits.npy")
y_val = val["type"].map({t: i for i, t in enumerate(TYPES)}).to_numpy()
assert (TYPES[logits.argmax(1)] == epoch_preds[CHOSEN_EPOCH]["pred"]).all(), "logits do not match the saved predictions"

def softmax_t(z, t):
    z = z / t - (z / t).max(1, keepdims=True)
    return np.exp(z) / np.exp(z).sum(1, keepdims=True)

def fit_temperature(z, y):  # same objective as Laya's fit_temperature: minimise log-loss over log T
    zt, yt = torch.tensor(z, dtype=torch.float64), torch.tensor(y)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=500)
    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(zt / log_t.exp(), yt)
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_t.exp().clamp(0.5, 5.0))  # Laya serves temperatures in [0.5, 5]

oof = np.zeros_like(logits)
fold_t = []
for tr, te in GroupKFold(5).split(logits, y_val, val["family_id"]):
    fold_t.append(fit_temperature(logits[tr], y_val[tr]))
    oof[te] = softmax_t(logits[te], fold_t[-1])
raw = softmax_t(logits, 1.0)
calib = pd.DataFrame({name: {"log-loss": log_loss(y_val, p, labels=range(4)),
                             "ECE": expected_calibration_error(p.max(1), TYPES[p.argmax(1)] == val["type"].to_numpy())}
                      for name, p in {"raw": raw, "temperature (out of fold)": oof}.items()}).T
KEEP_T = calib.loc["temperature (out of fold)", "ECE"] < calib.loc["raw", "ECE"] and \
         calib.loc["temperature (out of fold)", "log-loss"] <= calib.loc["raw", "log-loss"]
TEMPERATURE = fit_temperature(logits, y_val) if KEEP_T else 1.0
print(f"Fold temperatures: {[round(t, 2) for t in fold_t]} → final T = {TEMPERATURE:.3f} · keep: {KEEP_T}")

chosen_dir = Path(f"checkpoints/seed{SEED}/chosen_epoch{CHOSEN_EPOCH}_calibrated")
served_t = json.loads((chosen_dir / "rl_agent_config.json").read_text())["temperature"][0]
assert abs(served_t - TEMPERATURE) < 1e-3, f"saved checkpoint serves T={served_t}, notebook fitted {TEMPERATURE}"
calib.round(4)

Fold temperatures: [3.63, 3.35, 3.57, 3.53, 3.46] → final T = 3.509 · keep: True


/var/folders/v1/8fyl2wv11b93xnbrgwysprw40000gn/T/ipykernel_16760/1092204869.py:23: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:821.)
  return float(log_t.exp().clamp(0.5, 5.0))  # Laya serves temperatures in [0.5, 5]


,log-loss,ECE
raw,1.2168,0.1784
temperature (out of fold),0.5294,0.0759


In [14]:
# Phase 5 (compare) — every arm on validation, same score(); Arm D calibrated = softmax(logits / T)
pred_d = save_predictions(f"arm_d_seed{SEED}_calibrated", val, softmax_t(logits, TEMPERATURE))
preds[f"D · Laya fine-tuned, epoch {CHOSEN_EPOCH}, raw"] = epoch_preds[CHOSEN_EPOCH]
preds[f"D · Laya fine-tuned, epoch {CHOSEN_EPOCH}, calibrated"] = pred_d
final = pd.DataFrame({name: score(p) for name, p in preds.items()}).T
final.round(3)

,accuracy,macro F1,recall Incident,recall Request,recall Problem,recall Change,ECE,auto-routed @0.75,accuracy @0.75,own line for 95%,auto-routed @own line
A · Logistic Regression,0.790,0.807,0.712,0.991,0.577,0.959,0.045,0.485,0.956,0.75,0.485
"B · Laya zero-shot, raw",0.792,0.784,0.878,0.943,0.332,0.963,0.133,0.293,0.994,0.65,0.424
"C · Laya zero-shot, cleaned",0.789,0.783,0.870,0.943,0.334,0.963,0.131,0.292,0.994,0.65,0.421
"D · Laya fine-tuned, epoch 3, raw",0.799,0.810,0.774,0.994,0.485,0.984,0.178,0.965,0.809,NaN,0.000
"D · Laya fine-tuned, epoch 3, calibrated",0.799,0.810,0.774,0.994,0.485,0.984,0.082,0.807,0.858,0.90,0.480


*Calibration keeps the answers (accuracy 79.9%) and fixes most of the overconfidence: out of fold, ECE 0.178 → 0.076 and log-loss 1.22 → 0.53 (the table above shows 0.082: there the final T is scored on the same tickets it was fitted to, so the out-of-fold number is the one to quote), with T = 3.51 (all five folds 3.35–3.63). Calibrated, Arm D needs a 0.90 confidence line to route at ≥ 95% accuracy and then routes ~48% of tickets — the same share as the baseline at its 0.75 line. The 0.90 line is chosen on validation, so it is a logged selection step.*

### Where Arm D stands (validation, seed 42 only)
Accuracy 79.9% vs 79.0% — **+0.9 points, inside the noise band: a tie so far.** Fine-tuning moved the Incident/Problem line (Problem recall 0.33 zero-shot → 0.48) but not past the baseline's 0.58, and its calibration (0.076) is still behind the baseline's (0.045). **Next:** seeds 43 and 44 with the same settings, then the test set is opened once for A, B and D.